In [1]:
import tqdm 
import sys
import pandas as pd
sys.path.append('../kaggle_prediction_library/') 
from web_scraping.torvik_player_scraping_functions import get_html_from_torvik_players, get_data_from_html

### Scrape the Data

In [2]:
# note this is the day before the first four, for safety
# day_before_tournament_start_dict = {
#     '2021': '20210318',
#     '2022': '20220315',
#     '2023': '20230314', 
#     '2024': '20240319'
# }

day_before_tournament_start_dict = {
    '2025': '20250316',
}

day_before_tournament_start_df = pd.DataFrame(list(day_before_tournament_start_dict.items()), columns=['year', 'day_before_tourney_start'])
day_before_tournament_start_df['season_start_date'] = (day_before_tournament_start_df['year'].astype(int) - 1).astype(str) + '1101'


In [3]:
all_dfs = []

for index, row in tqdm.tqdm(day_before_tournament_start_df.iterrows()):
    
    start = row["season_start_date"]
    end = row["day_before_tourney_start"]
    year = row["year"]

    html = get_html_from_torvik_players(year, start, end, 40, womens=True)
    tmp_df = get_data_from_html(html)
    tmp_df["Season"] = year

    all_dfs.append(tmp_df)


0it [00:00, ?it/s]

Loading URL: https://barttorvik.com/ncaaw/playerstat.php?link=y&sIndex=53&minmin=5&year=2025&start=20241101&end=20250316


The chromedriver version (133.0.6943.53) detected in PATH at /opt/homebrew/bin/chromedriver might not be compatible with the detected chrome version (134.0.6998.89); currently, chromedriver 134.0.6998.88 is recommended for chrome 134.*, so it is advised to delete the driver in PATH and retry


Clicked on 'Games' column successfully.
'Show 100 more' clicked (1/40)
'Show 100 more' clicked (2/40)
'Show 100 more' clicked (3/40)
'Show 100 more' clicked (4/40)
'Show 100 more' clicked (5/40)
'Show 100 more' clicked (6/40)
'Show 100 more' clicked (7/40)
'Show 100 more' clicked (8/40)
'Show 100 more' clicked (9/40)
'Show 100 more' clicked (10/40)
'Show 100 more' clicked (11/40)
'Show 100 more' clicked (12/40)
'Show 100 more' clicked (13/40)
'Show 100 more' clicked (14/40)
'Show 100 more' clicked (15/40)
'Show 100 more' clicked (16/40)
'Show 100 more' clicked (17/40)
'Show 100 more' clicked (18/40)
'Show 100 more' clicked (19/40)
'Show 100 more' clicked (20/40)
'Show 100 more' clicked (21/40)
'Show 100 more' clicked (22/40)
'Show 100 more' clicked (23/40)
'Show 100 more' clicked (24/40)
'Show 100 more' clicked (25/40)
'Show 100 more' clicked (26/40)
'Show 100 more' clicked (27/40)
'Show 100 more' clicked (28/40)
'Show 100 more' clicked (29/40)
'Show 100 more' clicked (30/40)
'Show 100

1it [07:31, 451.15s/it]


In [4]:
final_df = pd.concat(all_dfs, axis=0)
final_df = final_df[final_df["Min%"].notnull()]


### Map to the Kaggle Ids 

In [67]:
final_kaggle_torvik_mapping = pd.read_csv("../data/sky_data/mappings/kaggle_torvik_mapping.csv")

In [68]:
teams = pd.read_csv("../data/WTeams.csv")

In [69]:
final_df["Final_Torvik_Team"] = final_df["Team"]
final_df = final_df.merge(final_kaggle_torvik_mapping, how="inner", on=["Final_Torvik_Team"])

In [70]:
final_df["TeamName"] = final_df["Kaggle_Team"]

In [71]:
final_df = final_df.merge(teams[["TeamID", "TeamName"]], how="left", on=["TeamName"])

In [72]:
torvik_player_data = pd.read_csv("../data/sky_data/ncaaw_torvik_player_data_2021_2024.csv")

In [74]:
to_write = pd.concat([torvik_player_data, final_df], axis=0)

In [75]:
to_write.to_csv("../data/sky_data/ncaaw_torvik_player_data_2021_2025.csv")